# Monte Carlo simulations

In [ ]:
import numpy as np
import pandas as pd
import sys
import glob
import plotly.express as px
from pathlib import Path


project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.utils import load_config
from src.monte_carlo import run_simulation_grid
from src.metrics import summarize_df, rmse_diff_table
from src.figures import frontier_heatmap, estimator_metric_heatmap, frontier_panels, estimator_metric_panels, combined_theory_empirical_frontier, plot_dml_rmse_vs_residual_variance, plot_residual_variance_vs_kappa, plot_e_error_vs_alpha_d, plot_m_error_vs_alpha_y, plot_resid_var_vs_alpha_d

c:\Users\danil\anaconda3\envs\vcausal\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Running MC Sim

In [3]:
config = load_config("baseline")

df = run_simulation_grid(config, save_each=False, n_jobs=8)

alpha_y=1, alpha_d=1, kappa=2: 100%|██████████| 25/25 [00:10<00:00,  2.29it/s]


In [3]:
folder = Path().cwd() / "results" / "raw"

files = list(folder.glob("*.parquet"))


all_results = [pd.read_parquet(f) for f in files]
df = pd.concat(all_results, ignore_index=True)

In [3]:
df

,alpha_y,alpha_d,kappa,seed,overlap,residual_d_var,replication,tau_true,ols_tau_hat,ols_se,ols_ci_lower,ols_ci_upper,dml_tau_hat,dml_se,dml_ci_lower,dml_ci_upper,m_mse,e_mse,estimated_resid_var,treatmeant_var
0,0.0,0.0,0.5,42,0.236198,0.231094,0,2.5,2.477188,0.131153,2.220129,2.734248,2.430809,0.149451,2.137889,2.723728,0.719268,0.095242,0.250348,0.250354
1,0.0,0.0,0.5,43,0.235765,0.237892,1,2.5,2.333767,0.168508,2.003492,2.664043,2.165821,0.187450,1.798426,2.533216,0.943236,0.133471,0.237858,0.237868
2,0.0,0.0,0.5,44,0.236275,0.225898,2,2.5,2.617421,0.163624,2.296719,2.938123,2.771607,0.178095,2.422547,3.120667,0.930536,0.094747,0.249646,0.249652
3,0.0,0.0,0.5,45,0.236036,0.242075,3,2.5,2.446021,0.156578,2.139129,2.752913,2.556268,0.181022,2.201470,2.911065,0.785613,0.122204,0.245362,0.245362
4,0.0,0.0,0.5,46,0.236145,0.234385,4,2.5,2.390011,0.151100,2.093855,2.686166,2.617982,0.177712,2.269672,2.966292,0.726990,0.112770,0.246285,0.246313
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
670,1.0,1.0,3.0,62,0.114266,0.112692,20,2.5,2.873264,0.229420,2.423601,3.322927,2.286917,0.221325,1.853127,2.720707,1.000838,0.235096,0.171666,0.171666
671,1.0,1.0,3.0,63,0.127963,0.110965,21,2.5,2.485181,0.329713,1.838944,3.131418,2.228035,0.215687,1.805297,2.650773,0.967300,0.236942,0.177110,0.177189
672,1.0,1.0,3.0,64,0.124030,0.111768,22,2.5,2.062501,0.327628,1.420349,2.704652,1.624917,0.258939,1.117407,2.132428,1.248123,0.248913,0.180376,0.180377
673,1.0,1.0,3.0,65,0.116676,0.125425,23,2.5,2.379633,0.261837,1.866432,2.892834,2.130467,0.258135,1.624532,2.636402,1.029315,0.239965,0.173505,0.173514


In [4]:
summary_df = summarize_df(df)
summary_df.head()

,estimator,bias,sd,rmse,coverage,avg_ci_length,avg_overlap,avg_residual_var,alpha_y,alpha_d,kappa
0,OLS,-0.044246,0.096333,0.104242,1.0,0.602621,0.236043,0.233597,0.0,0.0,0.5
1,DML,-0.009410,0.142303,0.139745,1.0,0.701002,0.236043,0.233597,0.0,0.0,0.5
2,OLS,-0.034438,0.136860,0.138446,1.0,0.634230,0.206591,0.206598,0.0,0.0,1.0
3,DML,-0.020713,0.180820,0.178374,1.0,0.735968,0.206591,0.206598,0.0,0.0,1.0
4,OLS,-0.038296,0.198515,0.198238,1.0,0.813278,0.114127,0.110698,0.0,0.0,3.0


## Frontiers


In [6]:
fig = frontier_heatmap(summary_df, kappa=0.5)
fig.show()

In [7]:
fig = estimator_metric_heatmap(summary_df, metric="bias", estimator="OLS", kappa=1)
fig.show()


In [8]:
fig = estimator_metric_heatmap(summary_df, metric="bias", estimator="DML", kappa=1)
fig.show()

In [9]:
fig = estimator_metric_heatmap(summary_df, metric="sd", estimator="DML", kappa=1)
fig.show()

In [10]:
fig = estimator_metric_heatmap(summary_df, metric="sd", estimator="OLS", kappa=1)
fig.show()

In [11]:
fig = estimator_metric_heatmap(summary_df, metric="coverage", estimator="DML", kappa=1)
fig.show()

In [12]:
fig = estimator_metric_heatmap(summary_df, metric="coverage", estimator="OLS", kappa=1)
fig.show()

In [13]:
fig = frontier_panels(summary_df, kappas=[0.5, 1, 2.0])
fig.show()

In [14]:
fig = estimator_metric_panels(summary_df, metric="bias", estimator="OLS", kappas=[ 0.5, 1.0, 2])
fig.show()

In [15]:
fig = estimator_metric_panels(summary_df, metric="bias", estimator="DML", kappas=[ 0.5, 1.0, 2])
fig.show()

In [16]:
fig = estimator_metric_panels(summary_df, metric="sd", estimator="OLS", kappas=[ 0.5, 1.0, 2])
fig.show()

In [17]:
fig = estimator_metric_panels(summary_df, metric="sd", estimator="DML", kappas=[ 0.5, 1.0, 2])
fig.show()

In [18]:
fig = estimator_metric_panels(summary_df, metric="overlap", estimator="OLS", kappas=[ 0.5, 1.0, 2])
fig.show()

In [19]:
fig = estimator_metric_panels(summary_df, metric="residual", estimator="DML", kappas=[ 0.5, 1.0, 2])
fig.show()

# Contour plots

In [20]:
rmse_dif = rmse_diff_table(summary_df)

In [21]:
fig = combined_theory_empirical_frontier(rmse_dif)
fig.show()

## Testing graphs

In [22]:
fig2 = plot_residual_variance_vs_kappa(
    df,
    resid_var_col="residual_d_var",
    kappa_col="kappa",
    alpha_d_col="alpha_d",
    alpha_y_col="alpha_y",
    agg="mean",
    average_over_alpha_y=True,
    title="Residualized Treatment Variance vs Overlap Regime"
)
fig2.show()

In [23]:
fig = plot_dml_rmse_vs_residual_variance(
    df=df,
    tau_hat_col="dml_tau_hat",
    tau_true_col="tau_true",
    resid_var_col="residual_d_var",
    alpha_y_col="alpha_y",
    alpha_d_col="alpha_d",
    kappa_col="kappa",
    average_over_alpha_d=True,
)
fig.show()

In [7]:
fig = plot_resid_var_vs_alpha_d(df, average_over_alpha_y=False)
fig.show()

In [19]:
fig1 = plot_m_error_vs_alpha_y(df)
fig1.show()

fig2 = plot_e_error_vs_alpha_d(df)
fig2.show()
